# Lot 2 : Alignement par Programmation Dynamique

Ce notebook permet de :
1. choisir deux reads (depuis le toy dataset) ou en saisir manuellement,
2. calculer la **Plus Longue Sous-Séquence Commune** (LCS) par programmation dynamique,
3. afficher le **score** d'alignement et la **position du chevauchement**,
4. afficher l'alignement de façon lisible.

In [ ]:
import sys
import os

sys.path.insert(0, os.path.abspath(".."))

from src.io_sequences import lire_fastq
from src.alignement import lcs, score_alignement, position_chevauchement, afficher_alignement

if not os.path.exists("../data/toy_reads.fastq"):
    from src.generer_toy import generer_toy
    generer_toy()

reads = lire_fastq("../data/toy_reads.fastq")
print("Nombre de reads disponibles :", len(reads))

Nombre de reads disponibles : 2000


## 1. Exemple simple (pour comprendre)

On commence avec deux petites séquences pour voir le résultat clairement.

In [ ]:
a = "ACGTACGT"
b = "ACGTTCGT"

longueur, sous_sequence, dp = lcs(a, b)
print("Read A :", a)
print("Read B :", b)
print("Score (longueur LCS) :", longueur)
print("Sous-sequence commune :", sous_sequence)
print()
print("Alignement :")
print(afficher_alignement(a, b))

Read A : ACGTACGT
Read B : ACGTTCGT
Score (longueur LCS) : 7
Sous-sequence commune : ACGTCGT

Alignement :
ACGTA-CGT
||||  |||
ACGT-TCGT


## 2. Aligner deux reads du jeu de données

On choisit deux reads par leur indice. Comme les reads proviennent de la même
séquence de référence, ceux qui se chevauchent partagent une longue sous-séquence.

In [ ]:
# Choisir ici les indices des deux reads a comparer
indice_a = 0
indice_b = 1

read_a = reads[indice_a][1]
read_b = reads[indice_b][1]

print("Read", indice_a, ":", read_a)
print("Read", indice_b, ":", read_b)
print()

score = score_alignement(read_a, read_b)
debut_a, fin_a, debut_b, fin_b = position_chevauchement(read_a, read_b)

print("Score d'alignement (longueur LCS) :", score)
print("Chevauchement dans le read A : positions", debut_a, "a", fin_a)
print("Chevauchement dans le read B : positions", debut_b, "a", fin_b)
print()
print("Alignement lisible :")
print(afficher_alignement(read_a, read_b))

Read 0 : CGAATAGGGATATAGGCTACGACATGTGCGGCGACCCTTGCGACAGTGAC
Read 1 : TCGTGAATAACGCGACGGCTGAGACGAACGGCGCGTGAATTAAGCGCTTA

Score d'alignement (longueur LCS) : 33
Chevauchement dans le read A : positions 0 a 48
Chevauchement dans le read B : positions 1 a 49

Alignement lisible :
-CG--AATA--G-GGATATA-GGCT-A--CGA-CATGTGCG-GC-GACCC-TT--GCGACAGTG-AC
 ||  ||||  | | |     |||| |  ||| |  | ||| |  ||    ||  ||| |  |  | 
TCGTGAATAACGCG-A----CGGCTGAGACGAAC--G-GCGCG-TGA---ATTAAGCG-C--T-TA-


## 3. Chercher le read qui chevauche le mieux un read donné

On compare un read à plusieurs autres et on garde celui qui a le meilleur score.
C'est l'idée de base pour détecter les chevauchements lors d'un assemblage.

In [ ]:
reference_read = reads[0][1]

meilleur_indice = -1
meilleur_score = -1

# On teste les 200 premiers reads pour aller vite
for i in range(1, 200):
    candidat = reads[i][1]
    score = score_alignement(reference_read, candidat)
    if score > meilleur_score:
        meilleur_score = score
        meilleur_indice = i

print("Read de reference (indice 0) :", reference_read)
print("Meilleur chevauchement trouve : read", meilleur_indice, "avec score", meilleur_score)
print()
print(afficher_alignement(reference_read, reads[meilleur_indice][1]))

Read de reference (indice 0) : CGAATAGGGATATAGGCTACGACATGTGCGGCGACCCTTGCGACAGTGAC
Meilleur chevauchement trouve : read 137 avec score 49

CGAATAGGGATATAGGCTA-CGACATGTGCGGCGACCCTTGCGACAGTGAC
||||||||||||||||| | |||||||||||||||||||||||||||||||
CGAATAGGGATATAGGC-AACGACATGTGCGGCGACCCTTGCGACAGTGAC
